In [1]:
import pandas as pd
import requests
import re
from bs4 import BeautifulSoup as bs
from fake_useragent import UserAgent
import plotly.express as px
from collections import Counter
from konlpy.tag import Hannanum
from datetime import datetime
import time
import FinanceDataReader as fdr
from tqdm import tqdm
import datetime

ModuleNotFoundError: ignored

In [ ]:
import requests
from bs4 import BeautifulSoup

In [ ]:
!pip install beautifulsoup4

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
!pip install konlpy

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 52.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.3/465.3 kB 32.7 MB/s eta 0:00:00


In [ ]:
!pip install fake_useragent

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.0 MB/s eta 0:00:00


In [ ]:
!pip install finance-datareader

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
import warnings
warnings.filterwarnings(action='ignore')

hannanum = Hannanum()

In [ ]:

header = {"user-agent": "Mozilla/5.0"}

date_lst = []

start_date = '20221001'
end_date = '20221231'

current_date = datetime.datetime.strptime(start_date, "%Y%m%d")
end_date = datetime.datetime.strptime(end_date, "%Y%m%d")

while current_date <= end_date:
    date_lst.append(current_date.strftime("%Y%m%d"))
    current_date += datetime.timedelta(days=1)

df_list = []

for date in date_lst:
    for page in range(1, 11):
        url = f'https://news.naver.com/main/list.naver?mode=LS2D&mid=shm&sid2=260&sid1=101&date={date}&page={page}'
        html = requests.get(url, headers=header)
        soup = BeautifulSoup(html.text, 'html.parser')

        link_li = []

        ## href에서 type06_headline이 의미하는 게 속보기사여서 속보기사의 개수만큼만 뽑혀져나온 것
        for i in range(1, 11):
            links = soup.select(f'#main_content > div.list_body.newsflash_body > ul.type06_headline > li:nth-of-type({i}) > dl > dt > a')
            if len(links) > 0:
                link_li.append(links[0].attrs['href'])

        for i in range(1, 11):
            link_list = soup.select(f'#main_content > div.list_body.newsflash_body > ul.type06 > li:nth-of-type({i}) > dl > dt > a')
            if len(link_list) > 0:
                link = link_list[0].attrs['href']
                link_li.append(link)

        title_lst = []
        description_lst = []
        date_lst = []

        for link in link_li:
            html = requests.get(link, headers=header)
            soup = BeautifulSoup(html.text, 'html.parser')

            title = soup.find('h2', {'id': 'title_area'})
            if title is not None:
                print(title)
                title_lst.append(title.text)

            description = soup.find('div', {'id': 'dic_area'})
            if description is not None:
                description = description.text
                description_lst.append(description)
            else:
                pass

            date = soup.find('span', {'class': 'media_end_head_info_datestamp_time _ARTICLE_DATE_TIME'})
            if date is not None:
                date = date['data-date-time']
                date_lst.append(date)

        df = pd.DataFrame(zip(date_lst, title_lst, description_lst, link_li), columns=['date', 'title', 'description', 'url'])
        df_list.append(df)

result_df_3 = pd.concat(df_list, ignore_index=True)

In [ ]:
result_df